# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditi-avni/ML-FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### My rule

Prioritize pages for review when they show signs of declining search performance and have enough historical search activity to make the signal useful.

The score gives more weight to:
- stronger recent decline
- worse current search position
- higher search visibility

### Reason codes

- `DECLINING` — page has a negative trend
- `LOW_POSITION` — page has a relatively poor average position
- `HIGH_VISIBILITY` — page has substantial impressions
- `REVIEW` — page has enough signals to justify manual review

The score is a prioritization rule, not proof that a page needs a specific SEO action.


In [16]:
import numpy as np
import pandas as pd

score_df = df.copy()

# Decline magnitude: larger value = stronger decline
score_df["decline_score"] = (
    score_df["trend_pct"].clip(upper=0).abs()
)

# Log-transform impressions
score_df["visibility_score"] = np.log1p(
    score_df["impressions_90d"]
)

# avg_position = 0 means no position data
position = score_df["avg_position"].replace(0, np.nan)

score_df["position_score"] = position.fillna(
    position.median()
)

# Normalize to 0-1
def minmax(series):
    if series.max() == series.min():
        return pd.Series(0, index=series.index)
    return (series - series.min()) / (series.max() - series.min())

score_df["decline_norm"] = minmax(
    score_df["decline_score"]
)

score_df["visibility_norm"] = minmax(
    score_df["visibility_score"]
)

score_df["position_norm"] = minmax(
    score_df["position_score"]
)

# Final baseline score
score_df["action_score"] = (
    0.50 * score_df["decline_norm"]
    + 0.30 * score_df["position_norm"]
    + 0.20 * score_df["visibility_norm"]
)

# Reason codes
visibility_threshold = score_df["impressions_90d"].median()

def get_reason(row):
    reasons = []

    if row["trend_pct"] < 0:
        reasons.append("DECLINING")

    if row["avg_position"] > 10:
        reasons.append("LOW_POSITION")

    if row["impressions_90d"] >= visibility_threshold:
        reasons.append("HIGH_VISIBILITY")

    if not reasons:
        reasons.append("REVIEW")

    return "|".join(reasons)

score_df["reason_code"] = score_df.apply(
    get_reason,
    axis=1
)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [17]:
# Rank all pages
score_df = score_df.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

score_df["rank"] = score_df.index + 1

# Create ranked queue
queue = score_df[
    [
        "rank",
        "content_id",
        "action_score",
        "reason_code",
        "trend_pct",
        "avg_position",
        "impressions_90d",
        "clicks_90d",
        "content_type",
        "main_intent"
    ]
].copy()

# Make sure output folder exists
import os

os.makedirs("work/outputs", exist_ok=True)

# Save CSV
output_path = "work/outputs/baseline_action_score.csv"

queue.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(queue))

queue.head(20)

Saved: work/outputs/baseline_action_score.csv
Rows: 30000


,rank,content_id,action_score,reason_code,trend_pct,avg_position,impressions_90d,clicks_90d,content_type,main_intent
0,1,content_939b70207d30,0.689895,DECLINING|LOW_POSITION,-99.5,80.5,695,1,keyword article,informational
1,2,content_798197311609,0.688564,DECLINING|LOW_POSITION|HIGH_VISIBILITY,-96.0,88.9,1003,0,keyword article,informational
2,3,content_742a8fcba2fe,0.679636,DECLINING|LOW_POSITION,-98.8,76.0,643,0,keyword article,informational
3,4,content_1f7b831ee01f,0.676826,DECLINING|LOW_POSITION|HIGH_VISIBILITY,-97.8,69.1,1249,0,keyword article,informational
4,5,content_308516bb0d31,0.671993,DECLINING|LOW_POSITION|HIGH_VISIBILITY,-95.5,69.1,1893,0,feedly article,NaN
5,6,content_76e3bdbef3e6,0.670443,DECLINING|LOW_POSITION,-100.0,70.1,391,0,keyword article,informational
6,7,content_b7e5b7cf98e3,0.670126,DECLINING|LOW_POSITION,-100.0,76.6,233,0,keyword article,transactional
7,8,content_5fca9d488e6b,0.669232,DECLINING|LOW_POSITION|HIGH_VISIBILITY,-93.8,79.6,1214,0,keyword article,informational
8,9,content_8409beb4f2ac,0.669142,DECLINING|LOW_POSITION,-99.6,64.9,608,0,keyword article,informational
9,10,content_20e876d26019,0.668726,DECLINING|LOW_POSITION|HIGH_VISIBILITY,-94.0,27.3,59949,11,keyword article,informational


## 3. Top-20 review

### Top-20 review

The top 20 pages are treated as manual-review candidates.

The baseline score only prioritizes pages for review; it does not determine the correct SEO action.

Confidence is based on the strength of the observed signals. A page could still be incorrectly prioritized because of seasonality, temporary search changes, measurement variation, or missing context.


In [18]:
top20 = queue.head(20).copy()

def confidence_note(row):

    if (
        row["trend_pct"] <= -30
        and row["impressions_90d"] >= visibility_threshold
    ):
        return "Higher confidence: strong observed decline with meaningful visibility"

    elif row["trend_pct"] < 0:
        return "Moderate confidence: observed decline, but manual checking is needed"

    else:
        return "Lower confidence: score driven by other observed signals"

top20["action"] = "MANUAL_REVIEW"

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_could_be_wrong"] = (
    "Temporary demand change, seasonality, "
    "measurement variation, or missing context"
)

display(
    top20[
        [
            "rank",
            "content_id",
            "action",
            "reason_code",
            "confidence_note",
            "what_could_be_wrong"
        ]
    ]
)

,rank,content_id,action,reason_code,confidence_note,what_could_be_wrong
0,1,content_939b70207d30,MANUAL_REVIEW,DECLINING|LOW_POSITION,"Moderate confidence: observed decline, but man...","Temporary demand change, seasonality, measurem..."
1,2,content_798197311609,MANUAL_REVIEW,DECLINING|LOW_POSITION|HIGH_VISIBILITY,Higher confidence: strong observed decline wit...,"Temporary demand change, seasonality, measurem..."
2,3,content_742a8fcba2fe,MANUAL_REVIEW,DECLINING|LOW_POSITION,"Moderate confidence: observed decline, but man...","Temporary demand change, seasonality, measurem..."
3,4,content_1f7b831ee01f,MANUAL_REVIEW,DECLINING|LOW_POSITION|HIGH_VISIBILITY,Higher confidence: strong observed decline wit...,"Temporary demand change, seasonality, measurem..."
4,5,content_308516bb0d31,MANUAL_REVIEW,DECLINING|LOW_POSITION|HIGH_VISIBILITY,Higher confidence: strong observed decline wit...,"Temporary demand change, seasonality, measurem..."
5,6,content_76e3bdbef3e6,MANUAL_REVIEW,DECLINING|LOW_POSITION,"Moderate confidence: observed decline, but man...","Temporary demand change, seasonality, measurem..."
6,7,content_b7e5b7cf98e3,MANUAL_REVIEW,DECLINING|LOW_POSITION,"Moderate confidence: observed decline, but man...","Temporary demand change, seasonality, measurem..."
7,8,content_5fca9d488e6b,MANUAL_REVIEW,DECLINING|LOW_POSITION|HIGH_VISIBILITY,Higher confidence: strong observed decline wit...,"Temporary demand change, seasonality, measurem..."
8,9,content_8409beb4f2ac,MANUAL_REVIEW,DECLINING|LOW_POSITION,"Moderate confidence: observed decline, but man...","Temporary demand change, seasonality, measurem..."
9,10,content_20e876d26019,MANUAL_REVIEW,DECLINING|LOW_POSITION|HIGH_VISIBILITY,Higher confidence: strong observed decline wit...,"Temporary demand change, seasonality, measurem..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [19]:
# Show weakest 10 picks
weak_picks = queue.tail(10)

print("Weak picks:")

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "action_score",
            "reason_code",
            "trend_pct",
            "avg_position",
            "impressions_90d"
        ]
    ]
)

# Check excluded / leakage columns
forbidden_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
]

print("\nLeakage / exclusion check:")

for col in forbidden_columns:
    print(
        f"{col}:",
        "PRESENT" if col in queue.columns else "NOT PRESENT"
    )

# Check source data for possible product/decision flags
product_flag_terms = [
    "action",
    "recommendation",
    "decision",
    "priority",
    "flag"
]

product_like_columns = [
    col for col in df.columns
    if any(term in col.lower() for term in product_flag_terms)
]

print("\nPotential product/decision-derived columns:")
print(product_like_columns)

print("\nScore components:")
print([
    "trend_pct",
    "avg_position",
    "impressions_90d"
])

print("\nNo future-window columns were added to the baseline score.")

Weak picks:


,rank,content_id,action_score,reason_code,trend_pct,avg_position,impressions_90d
29990,29991,content_0934cd438dd6,NaN,REVIEW,NaN,0.0,3
29991,29992,content_f452475bff46,NaN,REVIEW,NaN,0.8,15
29992,29993,content_cd850fb019b1,NaN,LOW_POSITION,NaN,15.2,5
29993,29994,content_b1d45033b059,NaN,LOW_POSITION,NaN,37.0,2
29994,29995,content_a3af3b8346d8,NaN,REVIEW,NaN,0.0,3
29995,29996,content_179533212cd0,NaN,LOW_POSITION|HIGH_VISIBILITY,NaN,11.8,1467
29996,29997,content_92a5d2709aa9,NaN,REVIEW,NaN,0.0,3
29997,29998,content_2dfd17269502,NaN,LOW_POSITION,NaN,10.3,3
29998,29999,content_23dce6a656e6,NaN,REVIEW,NaN,0.0,1
29999,30000,content_c322796023c8,NaN,REVIEW,NaN,0.0,1



Leakage / exclusion check:
content_id: PRESENT
client_id: NOT PRESENT
trend_direction: NOT PRESENT
trend_pct: PRESENT

Potential product/decision-derived columns:
[]

Score components:
['trend_pct', 'avg_position', 'impressions_90d']

No future-window columns were added to the baseline score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.